# set 模型探針：humanv0 主幹向量抽取

- 程式：`aicup/setmodel_trunk.py`。每局目標玩家第 31～180 手等距取 30 手，每手一個 768 維向量（`act_trunkfinal` 輸出的平均加最大值）。
- 外部資源：KataGo `python/`（tag `v1.15.0`，MIT）與 `b18c384nbt-humanv0.ckpt`。checkpoint 先找 Drive 快取，沒有才下載。
- Drive 的 `MyDrive/aicup-2026/training-v1.1/` 要有官方訓練集 v1.1 的 10 個 CSV（含 `HD[n]`）。`training/` 是舊版，不能用。
- 執行前把 `humansl-analysis/val-v180000-dev-p3.jsonl` 上傳到 `MyDrive/aicup-2026/setmodel/`（`eval` 用）。
- 輸出在 Drive 的 `MyDrive/aicup-2026/setmodel/`，每 200 局一個 `part-XXXXXX.npz`，中斷後重跑同一個 `MODE` 會接續。
- 速度由 CPU 核心數決定（編碼是純 Python），GPU 大多在等。

使用方式：改下一格的 `MODE` 後執行全部。依序 `eval` -> `train`（`bench` 只量速度，可略過）。開始前與結束後各記一次右上角資源面板的剩餘運算單元。


In [ ]:
MODE = 'eval'  # bench | eval | train
PLAYERS_PER_RANK = 0  # train 每級人數，0 表示每級全取
SKIP_PER_RANK = 350  # 去掉每級名單前 N 位（已在 trunk-train-350），0 表示不跳過

In [ ]:
import os, subprocess, shutil
from google.colab import drive, userdata

assert MODE in ('bench', 'eval', 'train'), MODE
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/aicup-2026'
OUT = DRIVE + '/setmodel'
TRAIN_DIR = DRIVE + '/training-v1.1'
REPO = '/content/aicup-2026-go'
KATAGO = '/content/KataGo'
KATAGO_COMMIT = 'c560d38585b446bc93b0a62c77364bc8e7d266a4'  # tag v1.15.0


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])
    return r.stdout.strip()


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('pull', '-q')
!pip install -q sgfmill
!git log --oneline -1
!nvidia-smi -L
!nproc
assert len([f for f in os.listdir(TRAIN_DIR) if f.startswith('train_') and f.endswith('.csv')]) == 10, TRAIN_DIR

if not os.path.exists(KATAGO + '/python/features.py'):
    git('clone', '-q', '--depth', '1', '--branch', 'v1.15.0', '--filter=blob:none', '--sparse',
        'https://github.com/lightvector/KataGo', KATAGO)
    git('-C', KATAGO, 'sparse-checkout', 'set', 'python')
assert git('-C', KATAGO, 'rev-parse', 'HEAD') == KATAGO_COMMIT, 'KataGo 的 commit 不是 v1.15.0'
os.environ['KATAGO_PYTHON'] = KATAGO + '/python'

# katagotraining.org 會擋 urllib，先用 Drive 快取，沒有才用 wget
HUMANV0 = '/content/models/b18c384nbt-humanv0.ckpt'
CACHE = DRIVE + '/katago/b18c384nbt-humanv0.ckpt'
os.makedirs(os.path.dirname(HUMANV0), exist_ok=True)
if not os.path.exists(HUMANV0):
    if os.path.exists(CACHE):
        shutil.copy(CACHE, HUMANV0)
    else:
        !wget -q -O {HUMANV0} https://media.katagotraining.org/uploaded/networks/models_extra/b18c384nbt-humanv0.ckpt
from aicup import katago_torch as kt
kt.ensure_humanv0(HUMANV0)  # 檢查大小與 SHA-256
if not os.path.exists(CACHE):
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    shutil.copy(HUMANV0, CACHE)
os.makedirs(OUT, exist_ok=True)
print('humanv0 %s，訓練集 %s，輸出 %s' % (HUMANV0, TRAIN_DIR, OUT))

In [ ]:
BASE = 'python -m aicup.setmodel_trunk %s --train-dir ' + TRAIN_DIR + ' --checkpoint ' + HUMANV0
if MODE == 'bench':
    cmd = BASE % 'bench' + ' --set train --positions 1000'
elif MODE == 'eval':
    keys = OUT + '/val-v180000-dev-p3.jsonl'
    assert os.path.exists(keys), '先把 val-v180000-dev-p3.jsonl 上傳到 ' + OUT
    cmd = BASE % 'extract' + ' --set eval --keys-from %s --out %s/trunk-eval' % (keys, OUT)
else:
    name = 'new' if SKIP_PER_RANK else (PLAYERS_PER_RANK or 'all')
    cmd = BASE % 'extract' + ' --set train --players-per-rank %d --skip-per-rank %d --out %s/trunk-train-%s' % (
        PLAYERS_PER_RANK, SKIP_PER_RANK, OUT, name)
print(cmd)
!{cmd}

In [ ]:
# 結果摘要：part 數與 timing.json
import glob, json
for d in sorted(glob.glob(OUT + '/trunk-*')):
    parts = glob.glob(d + '/part-*.npz')
    t = {}
    if os.path.exists(d + '/timing.json'):
        with open(d + '/timing.json') as f:
            t = json.load(f)
    print(d, '%d 個 part' % len(parts), t.get('runs', [])[-1:])